# DubbSystem Colab Demo
This notebook installs DubbSystem from git, uploads an MP4, runs the dubbing pipeline, and previews the dubbed result.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dennymarcels/dubbsystem/blob/main/notebooks/colab_dubb_demo.ipynb)

## Before You Run
Update the repository URL in the next cell before executing the notebook. A GPU runtime is strongly recommended for transcription and voice cloning.
If you have reviewed and accepted the applicable Coqui XTTS terms, set `COQUI_TOS_AGREED = "1"` in the configuration cell to skip the interactive license prompt during model download.

In [ ]:
import sys
REPO_URL = "https://github.com/dennymarcels/dubbsystem.git"
TARGET_LANGUAGE = "en"
TRANSCRIPTION_MODEL = "large-v3"
TRANSLATION_MODEL = "facebook/nllb-200-1.3B"
VOICE_SAMPLE_SECONDS = 60
LOG_LEVEL = "INFO"
COQUI_TOS_AGREED = "1"  # Set to "1" only after you review and accept the applicable Coqui terms.
PYTHON_BIN = "python3.11" if sys.version_info >= (3, 12) else "python3"
print(f"Notebook runtime: {sys.version.split()[0]}")
print(f"Installer Python: {PYTHON_BIN}")
print(f"Transcription model: {TRANSCRIPTION_MODEL}")
print(f"Translation model: {TRANSLATION_MODEL}")

In [ ]:
!apt-get update -y
!apt-get install -y ffmpeg python3.11 python3.11-venv
!git clone "$REPO_URL" /content/DubbSystem
%cd /content/DubbSystem
!rm -rf .venv
!$PYTHON_BIN -m venv .venv
!./.venv/bin/python -m pip install --upgrade pip "setuptools<81" wheel
!./.venv/bin/python -m pip install -e .
!./.venv/bin/python -m pip install "setuptools<81"

## Upload an MP4
Use the file picker to upload one source MP4 video to the Colab runtime.

In [ ]:
from google.colab import files

In [ ]:
input_path = "/content/drive/MyDrive/Cursos IA Expert/MLOps/vídeos/1.1 WSL.mp4"
print(f"Uploaded: {input_path}")

In [ ]:
from pathlib import Path
input_file = Path(input_path)
output_file = input_file.with_name(f"{input_file.stem}_dubbed{input_file.suffix}")
print(f"Expected output: {output_file}")

## Run Dubbing
This invokes the packaged CLI with progress logging enabled.

In [ ]:
!COQUI_TOS_AGREED={COQUI_TOS_AGREED} ./.venv/bin/dubb "{input_path}" "{output_file}" --target-language "{TARGET_LANGUAGE}" --transcription-model "{TRANSCRIPTION_MODEL}" --translation-model "{TRANSLATION_MODEL}" --voice-sample-seconds {VOICE_SAMPLE_SECONDS} --log-level "{LOG_LEVEL}" --keep-temp

## Inspect Intermediate Transcript
The pipeline writes timestamped transcript data into the temporary working directory.

In [ ]:
transcript_path = input_file.parent / '.dubb_tmp' / input_file.stem / 'transcript.json'
print(transcript_path)
if transcript_path.exists():
    print(transcript_path.read_text(encoding='utf-8')[:2000])
else:
    print('Transcript artifact not found yet. Re-run the dubbing cell first, or check whether the pipeline failed before translation completed.')

## Preview the Result
Play the dubbed MP4 inline inside Colab.

In [ ]:
from IPython.display import Video
Video(str(output_file), embed=True)

## Download the Dubbed Video
Use this if you want to save the generated MP4 back to your machine.

In [ ]:
files.download(str(output_file))